### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="give_me_some_credit",
    dataset_year="2011",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/GiveMeSomeCredit/overview",
    download_description="""
We download the data from Kaggle and uzip it to a predefined folder.

mkdir -p local-data-warehouse/give_me_some_credit/ && cd local-data-warehouse/give_me_some_credit && kaggle competitions download -c GiveMeSomeCredit && cd ../../ && unzip local-data-warehouse/give_me_some_credit/GiveMeSomeCredit.zip -d local-data-warehouse/give_me_some_credit/ && rm local-data-warehouse/give_me_some_credit/GiveMeSomeCredit.zip
""",
    # References
    academic_reference_bibtex="""@misc{cukierski2011credit,
    author = {Credit Fusion and Will Cukierski},
    title = {Give Me Some Credit},
    year = {2011},
    howpublished = {url{https://kaggle.com/competitions/GiveMeSomeCredit}},
    note = {Kaggle}
}
""",
    academic_reference_bibtex_key="cukierski2011credit",
    license="Public",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature and its value to be more descriptive.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="FinancialDistressNextTwoYears",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="FinancialDistressNextTwoYears",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/cs-training.csv", index_col=0, na_values="NA")

target_feature = "FinancialDistressNextTwoYears"
df = df.rename(columns={"SeriousDlqin2yrs": target_feature})
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"})

cat_features = [
    "FinancialDistressNextTwoYears",
    "RevolvingUtilizationOfUnsecuredLines",
]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 150,000
Columns: 11
Use sampling: False (sample size: 150,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome', 'age', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfDependents', 'NumberOfTime60-89DaysPastDueNotWorse']
Rows remaining as candidates after top-10 filter: 1,000 (of 150,000)

#### Duplicate Report
Total duplicate rows: 609 (0.41% of dataset)
Duplicate rows ignoring target: 646 (0.43% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,FinancialDistressNextTwoYears,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,No,0.029690,57,0,0.283244,10121.0,7,0,1,0,0.0
1,No,0.000000,57,0,0.142562,7750.0,9,0,1,0,0.0
2,No,0.036569,48,0,0.236294,6000.0,6,0,2,0,3.0
3,No,1.018331,41,0,0.163138,4958.0,4,0,0,0,0.0
4,No,1.008799,49,0,3942.000000,NaN,10,0,1,0,0.0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,FinancialDistressNextTwoYears,category,0.0,0.00,2.0,"No, Yes"
1,RevolvingUtilizationOfUnsecuredLines,category,0.0,0.00,125728.0,"0.0, 1.0, 1.0, 0.9501, 0.008, 0.9541, 0.7131, 0.005, 0.046, 0.5389"
2,MonthlyIncome,float64,29731.0,19.82,13594.0,"5000.0, 4000.0, 6000.0, 3000.0, 0.0, 2500.0, 10000.0, 3500.0, 4500.0, 7000.0"
3,NumberOfDependents,float64,3924.0,2.62,13.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0"
4,DebtRatio,float64,0.0,0.00,114194.0,"0.0, 1.0, 4.0, 2.0, 3.0, 5.0, 9.0, 10.0, 7.0, 13.0"
5,age,int64,0.0,0.00,86.0,"49, 48, 50, 63, 47, 46, 53, 51, 52, 56"
6,NumberOfTime30-59DaysPastDueNotWorse,int64,0.0,0.00,16.0,"0, 1, 2, 3, 4, 5, 98, 6, 7, 8"
7,NumberOfOpenCreditLinesAndLoans,int64,0.0,0.00,58.0,"6, 7, 5, 8, 4, 9, 10, 3, 11, 12"
8,NumberOfTimes90DaysLate,int64,0.0,0.00,19.0,"0, 1, 2, 3, 4, 98, 5, 6, 7, 8"
9,NumberRealEstateLoansOrLines,int64,0.0,0.00,28.0,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,150000.0,52.295207,14.771866,0.0,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.452760,5.145951,0.0,58.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,98.0
NumberRealEstateLoansOrLines,150000.0,1.018240,1.129771,0.0,54.0
NumberOfTime60-89DaysPastDueNotWorse,150000.0,0.240387,4.155179,0.0,98.0
NumberOfDependents,146076.0,0.757222,1.115086,0.0,20.0


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column                               rank                            
FinancialDistressNextTwoYears        1              No  139974  93.32
                                     2             Yes   10026   6.68
RevolvingUtilizationOfUnsecuredLines 1             0.0   10878   7.25
                                     2       0.9999999   10256   6.84
                                     3             1.0      17   0.01
                                     4       0.9500998       8   0.01
                                     5     0.007984032       6   0.00

In [8]:
# Target Distribution
target_df

,count,pct
FinancialDistressNextTwoYears,,
No,139974,93.32
Yes,10026,6.68


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to give_me_some_credit/019d9cdd-4000-7593-9945-fae38e6aefeb
019d9cdd-4000-7593-9945-fae38e6aefeb
c36feaaffab3d0e5304cf9cf1dcb573e64c2b28680fcce9e8f718ab0313b7781
